# Generalization audit and model tuning

The previous model was selected on one validation allocation. This audit excludes different whole recording groups and repeats fitting and selection inside each fold. It compares a refitted Isolation Forest, the current gradient-boosting configuration and a tuned gradient-boosting procedure. The app's saved model is preserved.

Five outer folds provide assessment predictions. Within each outer-training pool, three inner folds choose alert settings and, for the tuned procedure, one of four declared model configurations. Every global scale and estimator is fitted only on its permitted training groups. The first 60 readings of each assessed source recording still establish its unlabeled startup reference and receive no assessment.

These are retrospective development results on the original train/validation pool. The dataset and model family were already inspected before this audit. Grouped cross-validation strengthens the comparison but does not create fresh data or establish performance on another machine. The old test CSVs are not read.

In [ ]:
import json

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from forge.config import PROJECT_ROOT

result = json.loads((PROJECT_ROOT / "docs/generalization-results.json").read_text())
names = {
    "isolation_forest": "Isolation Forest",
    "current_hgb": "Current gradient boosting",
    "tuned_hgb": "Tuned gradient boosting",
}
pool = pd.DataFrame(result["pool"])
pool[["group", "sources", "rows", "normal", "anomalous", "unknown"]]

## Recording groups and fold boundaries

Names such as `valve2/1` identify recordings, not input features. Eight sensor channels supply the model features. Files sharing observations belong to the same overlap group, so copies cannot appear on both sides of a fold boundary.

The pool has 23 recordings in 21 groups. One group contains the much longer source-normal recording. The group table above exposes that imbalance. The allocation below is fixed before fitting, and each group is assessed once.

In [ ]:
pd.DataFrame(
    [
        {
            "outer_fold": fold["fold"],
            "fitting_groups": fold["fit_groups"],
            "excluded_groups": fold["assessment_groups"],
        }
        for fold in result["fold_results"]
    ]
)

## Precision compared with recall

The tuned procedure reaches 33.7% precision at 89.5% recall, compared with 32.5% precision at 91.0% recall for the current configuration under this audit.

Tuning produces 1,104 fewer false-positive readings, but misses 127 more anomalous readings and produces 83 more separate false alert episodes. Those costs limit the case for activating a model with a small precision gain. The report includes all three target checks.

Precision is the share of alerted readings annotated anomalous. Recall is the share of anomalous readings that trigger alerts. A 100% precision value means no false-positive readings occurred among the alerts. It does not mean accuracy is 100% or that no anomalies were missed. Precision is displayed as zero when there are no alerts, with the alert count retained in the results.

The earlier 95.1% precision used a different allocation and selection setting. Compare the procedures below on their matched excluded groups, rather than interpreting the difference from 95.1% as a direct estimate of overfitting.

In [ ]:
pooled = pd.DataFrame(
    {names[name]: entry["pooled"] for name, entry in result["procedures"].items()}
).T
pooled[
    [
        "precision",
        "recall",
        "f1",
        "alerted_readings",
        "fp",
        "fn",
        "false_alarm_onsets",
        "false_onsets_per_available_normal_hour",
        "macro_group_precision",
        "macro_group_recall",
    ]
]

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=list(names.values()))
for column, (name, entry) in enumerate(result["procedures"].items(), start=1):
    m = entry["pooled"]
    matrix = [[m["tn"], m["fp"]], [m["fn"], m["tp"]]]
    fig.add_trace(
        go.Heatmap(
            z=matrix,
            x=["No alert", "Alert"],
            y=["Normal", "Anomaly"],
            text=matrix,
            texttemplate="%{text}",
            colorscale="Blues",
            showscale=False,
            zmin=0,
            zmax=int(pooled[["tn", "fp", "fn", "tp"]].to_numpy().max()),
        ),
        row=1,
        col=column,
    )
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=370, title="Outer-fold confusion counts: each group assessed once")
fig.show()

## Does tuning help consistently?

Pooled precision can hide difficult groups. These plots compare precision and recall in each outer fold, using the same excluded groups for all three procedures. The table also shows whether inner selection met the minimum recall requirements. A failed requirement invokes the declared fallback and remains visible in the audit.

The tuned procedure may choose different hyperparameters in different folds. That is expected when evaluating a tuning procedure. None of those choices uses the corresponding outer-assessment labels.

In [ ]:
fold_rows = []
for fold in result["fold_results"]:
    for name, outcome in fold["procedures"].items():
        m = outcome["assessment"]["pooled"]
        fold_rows.append(
            {
                "fold": fold["fold"],
                "procedure": names[name],
                "candidate": outcome["candidate"],
                "precision": m["precision"],
                "recall": m["recall"],
                "training_precision": outcome["training"]["precision"],
                "training_recall": outcome["training"]["recall"],
                "inner_status": outcome["selection"]["status"],
            }
        )
folds = pd.DataFrame(fold_rows)
fig = make_subplots(rows=1, cols=2, subplot_titles=["Precision", "Recall"])
for procedure, color in zip(names.values(), ["#4777b8", "#d58036", "#35866b"], strict=True):
    selected = folds.loc[folds.procedure.eq(procedure)]
    for column, metric in enumerate(["precision", "recall"], start=1):
        fig.add_trace(
            go.Scatter(
                x=selected.fold,
                y=selected[metric],
                name=procedure,
                legendgroup=procedure,
                showlegend=column == 1,
                mode="lines+markers",
                line={"color": color},
                marker={"color": color},
            ),
            row=1,
            col=column,
        )
fig.update_yaxes(range=[0, 1])
fig.update_xaxes(title_text="Outer fold", dtick=1)
fig.update_layout(height=400, legend={"orientation": "h", "y": -0.25})
fig.show()
folds

## Training results and excluded-group results

Each training score uses the same fitted estimator and threshold as its outer assessment. Training readings were used for fitting, so these scores are expected to be optimistic. A large gap is a warning about fit or operating-condition sensitivity, but it cannot distinguish those causes by itself. The table describes the observed folds and does not provide confidence intervals.

In [ ]:
folds.groupby("procedure")[["training_precision", "precision", "training_recall", "recall"]].mean()

## Find the groups behind the aggregate

The following table pairs both gradient-boosting procedures on every group. Precision alone is insufficient: inspect false-positive and missed-anomaly counts together with recall. Initialization readings are unscored, but startup anomalies still count as misses. Available-normal FPR and alert frequency exclude unscored normal exposure.

In [ ]:
group_rows = []
for name, entry in result["procedures"].items():
    for group, m in entry["per_group"].items():
        group_rows.append(
            {
                "procedure": names[name],
                "group": group,
                "precision": m["precision"],
                "recall": m["recall"],
                "tp": m["tp"],
                "fp": m["fp"],
                "fn": m["fn"],
                "tn": m["tn"],
                "initialization_readings": m["unavailable_rows"],
                "false_onsets": m["false_alarm_onsets"],
            }
        )
group_results = pd.DataFrame(group_rows)
group_results.pivot(index="group", columns="procedure", values=["precision", "recall", "fp", "fn"])

## Final tuning artifact and remaining evidence

After the nested assessment, a separate five-fold grouped selection across the full development pool chooses final settings and refits a research artifact. Its selection scores are not independent evaluation. The artifact is saved locally and is not activated. The app's prior model and baseline remain unchanged.

The current data influenced the feature design before this audit. Further tuning against the outer results would make them another selection set. A stronger generalization assessment needs new recordings, with settings frozen before scoring. An unlabeled reference from an already faulty startup remains a limitation.

This notebook reads saved aggregate results only. See the [report](../docs/generalization-results.md) and [protocol](../docs/generalization-protocol.md) for commands, hashes and selection rules.

In [ ]:
final = result["final_refit"]
pd.Series(
    {
        "candidate": final["candidate"],
        "threshold": final["selection"]["threshold"],
        "persistence": final["selection"]["policy"]["persistence"],
        "selection_status": final["selection"]["status"],
        "fit_rows": final["fit_audit"]["fit_rows"],
        "activated": final["activated"],
        "model_sha256": final["model_sha256"],
    }
)